# Bonus B6 — Detectar transacciones atípicas y puntuar en línea

**Objetivo.** Recorrer el ciclo de vida de un modelo cuyo uso es **en línea**: una decisión por transacción, en el momento en que ocurre. Se entrena un detector **sin etiquetas**, se mide con anomalías conocidas, se registra, se puntúa por lotes y **una transacción a la vez** por el punto de conexión (*endpoint*) del modelo, y se monitorea sin necesitar etiquetas.

**Qué cambia frente a B1–B4**

| Tema | B1–B4 (propensión de actividad) | B6 (transacciones atípicas) |
|---|---|---|
| Tipo de problema | Supervisado, con etiqueta del mes siguiente | **No supervisado**: no hay etiquetas de fraude |
| Grano | Una fila por cuenta y mes | **Una fila por transacción** |
| Uso | Por lotes, una vez al mes | Por lotes **y en línea** |
| Cómo se mide | AUC con la etiqueta real | **Anomalías inyectadas** (conocidas) en validación y prueba |
| Monitoreo | Deriva y desempeño real | **Tasa de marcadas** y deriva de las variables, que no necesitan etiquetas |

**Honestidad sobre la evaluación.** En un caso real hay pocos casos confirmados o ninguno. Aquí se **inyectan** anomalías de tres tipos (monto extremo, madrugada y ráfaga) en agosto y septiembre para poder medir; las métricas son de datos sintéticos y solo sirven para practicar. Los meses de entrenamiento **no** llevan anomalías inyectadas.

**Idea central: el mismo código para entrenar y para servir.** Las variables de una transacción (qué tan alto es su monto para su tipo, qué tan inusual es su canal para esa cuenta, cuánto pasó desde la anterior) se calculan con **una sola función**. Un servicio en línea necesita esas mismas variables al llamar al *endpoint*: si se calculan distinto, el modelo da resultados distintos. Los almacenes de variables (*feature stores*) existen para resolver esto; aquí se resuelve con la función compartida y una **tabla de referencia** que viaja con el modelo.

**Requisitos.** Sesión 3 completada (tablas plata). Funciona en Trial salvo el *endpoint*, que está en vista previa. `p_confirmar_reemplazo = 'SI'` se necesita para registrar el modelo y escribir las tablas. Cada ejecución registra **una versión nueva** del modelo `<prefijo>_modelo_atipicas_tx`.


In [ ]:
p_lakehouse_esperado = ''
p_prefijo = ''
p_confirmar_reemplazo = 'NO'
p_tasa_revision = 0.01
p_probar_endpoint = 'SI'


In [ ]:
"""Caso de ML del bonus: propension de alta actividad de una cuenta bancaria.

Pregunta: dado el comportamiento de una cuenta hasta el cierre de un mes, ¿tendra
al menos `UMBRAL_ALTA_ACTIVIDAD` transacciones validas el mes siguiente?

Los datos son sinteticos: las metricas sirven para practicar el ciclo de vida,
no para estimar el rendimiento de un modelo real. Las funciones de datos y de
modelo usan pandas y scikit-learn, que forman parte del runtime de Fabric. La
seccion final (ciclo MLOps) recibe `spark` y `mlflow` como argumentos en lugar de
importarlos, para poder probarla fuera de Fabric.
"""

from __future__ import annotations

import json
import math
import re
from datetime import datetime, timezone
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd


PATRON_PREFIJO = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,19}\Z", re.ASCII)
UMBRAL_ALTA_ACTIVIDAD = 9
ETIQUETA = "alta_actividad_siguiente"
CLAVE = "cuenta_id"
CORTE = "corte"

CANALES = ("BANCA_MOVIL", "WEB", "ATM", "SUCURSAL")
# Todas las variables de comportamiento usan una ventana fija de dos meses (el mes de
# corte y el anterior). Las variables acumuladas o la antigüedad crecen con el
# calendario y producirian una deriva artificial en el monitoreo.
VENTANA_MESES = 2
FEATURES_NUMERICAS = (
    "tx_mes_corte",
    "tx_mes_previo",
    "tx_ventana",
    "monto_ventana",
    "monto_prom_ventana",
    "pct_credito",
    "pct_fin_semana",
    "pct_banca_movil",
    "pct_web",
    "pct_atm",
    "pct_sucursal",
    "n_tipos",
    "dias_desde_ultima",
    "saldo_inicial",
)
FEATURES_CATEGORICAS = ("producto_id", "segmento")
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
# Atributos que NO entran al modelo; se usan solo para auditar diferencias entre grupos.
VARIABLES_PARA_AUDITORIA = ("grupo_edad", "provincia")

CORTES_ENTRENAMIENTO = ("2026-05", "2026-06")
CORTE_VALIDACION = "2026-07"
CORTE_PRUEBA = "2026-08"
CORTE_PUNTUACION = "2026-09"

UMBRAL_PSI_MODERADO = 0.10
UMBRAL_PSI_ALTO = 0.25

_COLUMNAS_TRANSACCIONES = (
    "cuenta_id",
    "fecha_hora",
    "tipo_transaccion",
    "direccion",
    "canal",
    "monto",
)
_COLUMNAS_CUENTAS = ("cuenta_id", "cliente_id", "producto_id", "fecha_apertura", "saldo_inicial")
_COLUMNAS_CLIENTES = ("cliente_id", "segmento")


def _exigir_columnas(df: pd.DataFrame, columnas: Sequence[str], nombre: str) -> None:
    faltantes = [columna for columna in columnas if columna not in df.columns]
    if faltantes:
        raise ValueError(
            f"{nombre}: expected columns {list(columnas)!r}; observed missing {faltantes!r}"
        )


def _a_numerico(serie: pd.Series) -> pd.Series:
    # Spark entrega decimal(18,2) como objetos Decimal; se convierte a float.
    return pd.to_numeric(serie.astype(float), errors="raise")


def _fin_de_mes(periodo: pd.Period) -> pd.Timestamp:
    return periodo.end_time.normalize()


def construir_features(
    transacciones: pd.DataFrame,
    cuentas: pd.DataFrame,
    clientes: pd.DataFrame,
    corte: str,
    ultimo_dia_datos: str | None = None,
    umbral: int = UMBRAL_ALTA_ACTIVIDAD,
) -> pd.DataFrame:
    """Una fila por cuenta con el comportamiento de los dos ultimos meses hasta `corte` (AAAA-MM).

    Solo se usan movimientos con fecha dentro del mes de corte o el anterior. La
    etiqueta mira el mes siguiente y queda vacia (NaN) cuando ese mes no esta
    completo en los datos: son las filas que se puntuan, no las que se entrenan.
    `ultimo_dia_datos` (AAAA-MM-DD) indica hasta donde llegan los datos; si se omite,
    se toma la fecha del ultimo movimiento.
    """
    _exigir_columnas(transacciones, _COLUMNAS_TRANSACCIONES, "transacciones")
    _exigir_columnas(cuentas, _COLUMNAS_CUENTAS, "cuentas")
    _exigir_columnas(clientes, _COLUMNAS_CLIENTES, "clientes")

    corte_p = pd.Period(corte, freq="M")
    inicio_ventana = corte_p - (VENTANA_MESES - 1)
    tx = transacciones.loc[:, list(_COLUMNAS_TRANSACCIONES)].copy()
    tx["fecha_hora"] = pd.to_datetime(tx["fecha_hora"])
    if tx["fecha_hora"].dt.tz is not None:
        tx["fecha_hora"] = tx["fecha_hora"].dt.tz_localize(None)
    tx["monto"] = _a_numerico(tx["monto"])
    tx["periodo"] = tx["fecha_hora"].dt.to_period("M")

    ventana = tx.loc[(tx["periodo"] >= inicio_ventana) & (tx["periodo"] <= corte_p)].copy()
    ventana["es_credito"] = (ventana["direccion"] == "CREDITO").astype(float)
    ventana["fin_semana"] = (ventana["fecha_hora"].dt.day_of_week >= 5).astype(float)
    for canal in CANALES:
        ventana["pct_" + canal.lower()] = (ventana["canal"] == canal).astype(float)

    por_cuenta = ventana.groupby(CLAVE)
    features = pd.DataFrame(index=pd.Index(cuentas[CLAVE].unique(), name=CLAVE))
    features["tx_mes_corte"] = ventana.loc[ventana["periodo"] == corte_p].groupby(CLAVE).size()
    features["tx_mes_previo"] = ventana.loc[ventana["periodo"] == corte_p - 1].groupby(CLAVE).size()
    features["tx_ventana"] = por_cuenta.size()
    features["monto_ventana"] = por_cuenta["monto"].sum()
    features["monto_prom_ventana"] = por_cuenta["monto"].mean()
    features["pct_credito"] = por_cuenta["es_credito"].mean()
    features["pct_fin_semana"] = por_cuenta["fin_semana"].mean()
    for canal in CANALES:
        columna = "pct_" + canal.lower()
        features[columna] = por_cuenta[columna].mean()
    features["n_tipos"] = por_cuenta["tipo_transaccion"].nunique()

    fin_corte = _fin_de_mes(corte_p)
    dias_ventana = (fin_corte - inicio_ventana.start_time.normalize()).days
    features["dias_desde_ultima"] = (fin_corte - por_cuenta["fecha_hora"].max()).dt.days

    cero_si_vacio = [c for c in features.columns if c != "dias_desde_ultima"]
    features[cero_si_vacio] = features[cero_si_vacio].fillna(0.0)
    features["dias_desde_ultima"] = features["dias_desde_ultima"].fillna(dias_ventana)
    for columna in (
        "tx_mes_corte", "tx_mes_previo", "tx_ventana", "n_tipos", "dias_desde_ultima"
    ):
        features[columna] = features[columna].astype("int64")

    base = cuentas.loc[:, list(_COLUMNAS_CUENTAS)].drop_duplicates(CLAVE).merge(
        clientes.loc[:, list(_COLUMNAS_CLIENTES)], on="cliente_id", how="left", validate="m:1"
    )
    base["saldo_inicial"] = _a_numerico(base["saldo_inicial"])
    base = base.drop(columns=["cliente_id", "fecha_apertura"])

    resultado = features.reset_index().merge(base, on=CLAVE, how="left", validate="1:1")
    resultado.insert(1, CORTE, str(corte_p))

    siguiente = corte_p + 1
    ultimo_dia = (
        pd.Timestamp(ultimo_dia_datos).normalize()
        if ultimo_dia_datos is not None
        else tx["fecha_hora"].max().normalize()
    )
    if _fin_de_mes(siguiente) <= ultimo_dia:
        proximo = tx.loc[tx["periodo"] == siguiente].groupby(CLAVE).size()
        conteo = resultado[CLAVE].map(proximo).fillna(0)
        resultado[ETIQUETA] = (conteo >= umbral).astype("float64")
    else:
        resultado[ETIQUETA] = np.nan

    columnas = [CLAVE, CORTE, *FEATURES, ETIQUETA]
    return resultado.loc[:, columnas].sort_values(CLAVE).reset_index(drop=True)


def construir_dataset(
    transacciones: pd.DataFrame,
    cuentas: pd.DataFrame,
    clientes: pd.DataFrame,
    cortes: Sequence[str],
    ultimo_dia_datos: str | None = None,
    umbral: int = UMBRAL_ALTA_ACTIVIDAD,
) -> pd.DataFrame:
    partes = [
        construir_features(transacciones, cuentas, clientes, corte, ultimo_dia_datos, umbral)
        for corte in cortes
    ]
    return pd.concat(partes, ignore_index=True)


def validar_features(df: pd.DataFrame, cuentas_esperadas: int | None = None) -> list[str]:
    """Devuelve la lista de problemas; una lista vacia significa que pasa."""
    problemas: list[str] = []
    esperadas = [CLAVE, CORTE, *FEATURES, ETIQUETA]
    if list(df.columns) != esperadas:
        return [f"columns: expected {esperadas!r}; observed {list(df.columns)!r}"]
    if df.duplicated([CLAVE, CORTE]).any():
        problemas.append("expected one row per (cuenta_id, corte); observed duplicates")
    numericas = list(FEATURES_NUMERICAS)
    nulos = df[numericas + list(FEATURES_CATEGORICAS)].isna().sum()
    for columna, cantidad in nulos[nulos > 0].items():
        problemas.append(f"{columna}: expected no nulls; observed {int(cantidad)}")
    for columna in ("tx_mes_corte", "tx_mes_previo", "tx_ventana", "monto_ventana", "saldo_inicial"):
        if (df[columna] < 0).any():
            problemas.append(f"{columna}: expected non-negative values; observed negatives")
    for columna in ("pct_credito", "pct_fin_semana", *(f"pct_{c.lower()}" for c in CANALES)):
        if ((df[columna] < 0) | (df[columna] > 1)).any():
            problemas.append(f"{columna}: expected values between 0 and 1; observed outside")
    if not (df["tx_mes_corte"] + df["tx_mes_previo"] == df["tx_ventana"]).all():
        problemas.append("tx_mes_corte + tx_mes_previo = tx_ventana: expected to hold; observed violations")
    invalidas = df[ETIQUETA].dropna()
    if not invalidas.isin([0.0, 1.0]).all():
        problemas.append(f"{ETIQUETA}: expected 0, 1 or empty; observed other values")
    if cuentas_esperadas is not None:
        por_corte = df.groupby(CORTE).size()
        for corte, filas in por_corte.items():
            if filas != cuentas_esperadas:
                problemas.append(
                    f"corte {corte}: expected {cuentas_esperadas} rows; observed {int(filas)}"
                )
    return problemas


def dividir_temporal(
    df: pd.DataFrame,
    entrenamiento: Sequence[str] = CORTES_ENTRENAMIENTO,
    validacion: str = CORTE_VALIDACION,
    prueba: str = CORTE_PRUEBA,
    puntuacion: str = CORTE_PUNTUACION,
) -> dict[str, pd.DataFrame]:
    """Separa por tiempo, nunca al azar.

    Cada corte etiqueta el mes siguiente. Las etiquetas de entrenamiento deben
    ser anteriores a las de validacion, y estas a las de prueba, para que el
    modelo nunca vea el futuro que se usa para juzgarlo.
    """
    orden = [*sorted(entrenamiento), validacion, prueba, puntuacion]
    if orden != sorted(orden) or len(set(orden)) != len(orden):
        raise ValueError(
            f"cortes: expected strictly increasing training < validation < test < scoring; observed {orden!r}"
        )
    partes = {
        "entrenamiento": df[df[CORTE].isin(entrenamiento)],
        "validacion": df[df[CORTE] == validacion],
        "prueba": df[df[CORTE] == prueba],
        "puntuacion": df[df[CORTE] == puntuacion],
    }
    for nombre, parte in partes.items():
        if parte.empty:
            raise ValueError(f"{nombre}: expected at least one row; observed none")
        con_etiqueta = parte[ETIQUETA].notna().all()
        sin_etiqueta = parte[ETIQUETA].isna().all()
        if nombre == "puntuacion" and not sin_etiqueta:
            raise ValueError("puntuacion: expected an empty label; observed labelled rows")
        if nombre != "puntuacion" and not con_etiqueta:
            raise ValueError(f"{nombre}: expected labelled rows; observed missing labels")
    return {nombre: parte.reset_index(drop=True) for nombre, parte in partes.items()}


def crear_modelo(tipo: str, semilla: int = 42) -> Any:
    """Pipeline de scikit-learn: preparacion + modelo. `tipo`: logreg, rf o hgb."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    estimadores = {
        "logreg": lambda: LogisticRegression(max_iter=1000),
        "rf": lambda: RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, random_state=semilla, n_jobs=2
        ),
        "hgb": lambda: HistGradientBoostingClassifier(random_state=semilla),
    }
    if tipo not in estimadores:
        raise ValueError(f"tipo: expected one of {sorted(estimadores)!r}; observed {tipo!r}")
    preparacion = ColumnTransformer(
        [
            ("numericas", StandardScaler(), list(FEATURES_NUMERICAS)),
            ("categoricas", OneHotEncoder(handle_unknown="ignore"), list(FEATURES_CATEGORICAS)),
        ]
    )
    return Pipeline([("preparacion", preparacion), ("modelo", estimadores[tipo]())])


def evaluar_clasificador(y_real: Any, probabilidad: Any, umbral: float = 0.5) -> dict[str, float]:
    from sklearn.metrics import (
        average_precision_score,
        brier_score_loss,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    y = np.asarray(y_real, dtype=int)
    p = np.asarray(probabilidad, dtype=float)
    pred = (p >= umbral).astype(int)
    return {
        "auc": float(roc_auc_score(y, p)),
        "average_precision": float(average_precision_score(y, p)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "brier": float(brier_score_loss(y, p)),
        "prevalencia": float(y.mean()),
    }


def baseline_regla(df: pd.DataFrame, umbral: int = UMBRAL_ALTA_ACTIVIDAD) -> np.ndarray:
    """Regla sin aprendizaje: la actividad del mes de corte predice la del siguiente."""
    return (df["tx_mes_corte"].to_numpy() / float(umbral)).clip(0, 1)


def simular_deriva_actividad(df: pd.DataFrame, factor: float = 1.6) -> pd.DataFrame:
    """Copia de `df` con la actividad multiplicada por `factor` (por ejemplo, una campana).

    Sirve para comprobar que el monitoreo detecta un cambio de poblacion; no
    modifica las etiquetas.
    """
    if factor <= 0:
        raise ValueError(f"factor: expected a positive number; observed {factor!r}")
    simulado = df.copy()
    for columna in ("tx_mes_corte", "tx_mes_previo"):
        simulado[columna] = (simulado[columna] * factor).round().astype("int64")
    simulado["tx_ventana"] = simulado["tx_mes_corte"] + simulado["tx_mes_previo"]
    simulado["monto_ventana"] = simulado["monto_ventana"] * factor
    return simulado


def importancia_permutacion(
    modelo: Any, df: pd.DataFrame, repeticiones: int = 5, semilla: int = 42
) -> pd.DataFrame:
    from sklearn.inspection import permutation_importance

    resultado = permutation_importance(
        modelo,
        df[list(FEATURES)],
        df[ETIQUETA].astype(int),
        scoring="roc_auc",
        n_repeats=repeticiones,
        random_state=semilla,
    )
    tabla = pd.DataFrame(
        {
            "variable": list(FEATURES),
            "caida_auc": resultado.importances_mean,
            "desviacion": resultado.importances_std,
        }
    )
    return tabla.sort_values("caida_auc", ascending=False).reset_index(drop=True)


def metricas_por_grupo(
    y_real: Any, probabilidad: Any, grupo: Any, umbral: float = 0.5
) -> pd.DataFrame:
    """Auditoria por grupo (por ejemplo, grupo de edad), aunque no sea una variable del modelo."""
    from sklearn.metrics import roc_auc_score

    tabla = pd.DataFrame(
        {
            "y": np.asarray(y_real, dtype=int),
            "p": np.asarray(probabilidad, dtype=float),
            "grupo": np.asarray(grupo),
        }
    )
    filas = []
    for nombre, parte in tabla.groupby("grupo"):
        auc = float("nan")
        if parte["y"].nunique() == 2:
            auc = float(roc_auc_score(parte["y"], parte["p"]))
        filas.append(
            {
                "grupo": nombre,
                "filas": int(len(parte)),
                "prevalencia": float(parte["y"].mean()),
                "tasa_positivos_predichos": float((parte["p"] >= umbral).mean()),
                "auc": auc,
            }
        )
    return pd.DataFrame(filas)


def _proporciones(indices: np.ndarray, cantidad: int, epsilon: float) -> np.ndarray:
    conteos = np.bincount(indices, minlength=cantidad).astype(float)
    return np.clip(conteos / max(len(indices), 1), epsilon, None)


def psi_numerico(
    referencia: Any, actual: Any, particiones: int = 10, epsilon: float = 1e-4
) -> float:
    """Indice de estabilidad de la poblacion con cortes por cuantiles de la referencia."""
    ref = np.asarray(referencia, dtype=float)
    act = np.asarray(actual, dtype=float)
    if ref.size == 0 or act.size == 0:
        raise ValueError("psi_numerico: expected non-empty samples; observed an empty one")
    interiores = np.unique(np.quantile(ref, np.linspace(0, 1, particiones + 1)[1:-1]))
    cantidad = len(interiores) + 1
    # side="left": un valor igual a un corte queda en el intervalo de la izquierda; asi una
    # referencia constante tiene su propio intervalo y un cambio se detecta.
    p_ref = _proporciones(np.searchsorted(interiores, ref, side="left"), cantidad, epsilon)
    p_act = _proporciones(np.searchsorted(interiores, act, side="left"), cantidad, epsilon)
    return float(np.sum((p_act - p_ref) * np.log(p_act / p_ref)))


def psi_categorico(referencia: Any, actual: Any, epsilon: float = 1e-4) -> float:
    ref = pd.Series(referencia).astype(str)
    act = pd.Series(actual).astype(str)
    if ref.empty or act.empty:
        raise ValueError("psi_categorico: expected non-empty samples; observed an empty one")
    categorias = sorted(set(ref) | set(act))
    p_ref = np.clip(ref.value_counts(normalize=True).reindex(categorias).fillna(0).to_numpy(), epsilon, None)
    p_act = np.clip(act.value_counts(normalize=True).reindex(categorias).fillna(0).to_numpy(), epsilon, None)
    return float(np.sum((p_act - p_ref) * np.log(p_act / p_ref)))


def clasificar_psi(valor: float) -> str:
    if valor >= UMBRAL_PSI_ALTO:
        return "alta"
    if valor >= UMBRAL_PSI_MODERADO:
        return "moderada"
    return "estable"


def reporte_deriva(referencia: pd.DataFrame, actual: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for columna in FEATURES_NUMERICAS:
        valor = psi_numerico(referencia[columna], actual[columna])
        filas.append({"variable": columna, "tipo": "numerica", "psi": valor, "deriva": clasificar_psi(valor)})
    for columna in FEATURES_CATEGORICAS:
        valor = psi_categorico(referencia[columna], actual[columna])
        filas.append({"variable": columna, "tipo": "categorica", "psi": valor, "deriva": clasificar_psi(valor)})
    return pd.DataFrame(filas).sort_values("psi", ascending=False).reset_index(drop=True)


def decidir_promocion(
    retador: Mapping[str, float],
    campeon: Mapping[str, float] | None = None,
    metrica: str = "auc",
    margen_minimo: float = 0.005,
    piso: float = 0.70,
) -> dict[str, Any]:
    """Regla explicita de promocion: piso de calidad y mejora minima sobre el campeon."""
    valor = float(retador[metrica])
    if not math.isfinite(valor):
        return {"decision": "rechazar", "motivo": f"{metrica} is not finite", "mejora": None}
    if valor < piso:
        return {
            "decision": "rechazar",
            "motivo": f"{metrica} {valor:.4f} is below the floor {piso:.4f}",
            "mejora": None,
        }
    if campeon is None:
        return {"decision": "promover", "motivo": "there is no champion yet", "mejora": None}
    mejora = valor - float(campeon[metrica])
    if mejora >= margen_minimo:
        return {
            "decision": "promover",
            "motivo": f"{metrica} improves by {mejora:.4f} (minimum {margen_minimo:.4f})",
            "mejora": mejora,
        }
    return {
        "decision": "mantener",
        "motivo": f"{metrica} improves by {mejora:.4f}, below the minimum {margen_minimo:.4f}",
        "mejora": mejora,
    }


def evaluar_alertas(
    deriva: pd.DataFrame,
    auc_actual: float | None = None,
    auc_referencia: float | None = None,
    caida_auc_maxima: float = 0.05,
    variables_alta_maxima: int = 2,
) -> dict[str, Any]:
    """Resume la salud del modelo. La caida de AUC solo se conoce cuando llegan las etiquetas."""
    alertas: list[str] = []
    altas = deriva.loc[deriva["deriva"] == "alta", "variable"].tolist()
    if len(altas) > variables_alta_maxima:
        alertas.append(f"{len(altas)} variables con deriva alta: {altas}")
    if auc_actual is not None and auc_referencia is not None:
        caida = auc_referencia - auc_actual
        if caida > caida_auc_maxima:
            alertas.append(f"auc fell {caida:.4f} (maximum {caida_auc_maxima:.4f})")
    if any(a.startswith("auc fell") for a in alertas):
        recomendacion = "reentrenar"
    elif alertas:
        recomendacion = "revisar"
    else:
        recomendacion = "sin_accion"
    return {"alertas": alertas, "recomendacion": recomendacion}


def puntuar(modelo: Any, df: pd.DataFrame, umbral: float = 0.5) -> pd.DataFrame:
    probabilidad = modelo.predict_proba(df[list(FEATURES)])[:, 1]
    return pd.DataFrame(
        {
            CLAVE: df[CLAVE].to_numpy(),
            CORTE: df[CORTE].to_numpy(),
            "probabilidad_alta_actividad": probabilidad.astype(float),
            "prediccion": (probabilidad >= umbral).astype("int64"),
        }
    )


def a_filas(df: pd.DataFrame) -> list[tuple[Any, ...]]:
    """Filas con tipos nativos de Python y None en lugar de NaN, para crear un DataFrame de Spark."""
    filas = []
    for fila in df.itertuples(index=False, name=None):
        nativa = []
        for valor in fila:
            if isinstance(valor, float) and math.isnan(valor):
                nativa.append(None)
            elif isinstance(valor, np.generic):
                nativa.append(valor.item())
            else:
                nativa.append(valor)
        filas.append(tuple(nativa))
    return filas


# ---------------------------------------------------------------------------
# Ciclo MLOps: validar, entrenar un retador, decidir, puntuar y monitorear.
# Recibe `spark` y `mlflow` como argumentos; el modulo no los importa.
# ---------------------------------------------------------------------------

COLUMNAS_FEATURES = (
    (CLAVE, "string"),
    (CORTE, "string"),
    ("tx_mes_corte", "bigint"),
    ("tx_mes_previo", "bigint"),
    ("tx_ventana", "bigint"),
    ("monto_ventana", "double"),
    ("monto_prom_ventana", "double"),
    ("pct_credito", "double"),
    ("pct_fin_semana", "double"),
    ("pct_banca_movil", "double"),
    ("pct_web", "double"),
    ("pct_atm", "double"),
    ("pct_sucursal", "double"),
    ("n_tipos", "bigint"),
    ("dias_desde_ultima", "bigint"),
    ("saldo_inicial", "double"),
    ("producto_id", "string"),
    ("segmento", "string"),
    (ETIQUETA, "double"),
)
COLUMNAS_PUNTUACION = (
    (CLAVE, "string"),
    (CORTE, "string"),
    ("probabilidad_alta_actividad", "double"),
    ("prediccion", "bigint"),
    ("version_modelo", "bigint"),
    ("fecha_utc", "string"),
)
COLUMNAS_REGISTRO = (
    ("secuencia", "bigint"),
    ("fecha_utc", "string"),
    ("evento", "string"),
    ("version", "bigint"),
    ("tipo_modelo", "string"),
    ("auc_validacion", "double"),
    ("auc_prueba", "double"),
    ("corte_prueba", "string"),
    ("run_id", "string"),
    ("motivo", "string"),
)
COLUMNAS_DERIVA = (
    ("corte", "string"),
    ("variable", "string"),
    ("tipo", "string"),
    ("psi", "double"),
    ("deriva", "string"),
    ("simulada", "bigint"),
    ("fecha_utc", "string"),
)
COLUMNAS_DESEMPENO = (
    ("corte_prediccion", "string"),
    ("mes_evaluado", "string"),
    ("version_modelo", "bigint"),
    ("filas", "bigint"),
    ("auc", "double"),
    ("average_precision", "double"),
    ("prevalencia_real", "double"),
    ("fecha_utc", "string"),
)


def nombres_ml(prefijo: str) -> dict[str, str]:
    p = prefijo.lower()
    return {
        "features": f"{p}_gold_ml_features_cuentas",
        "puntuacion": f"{p}_gold_ml_puntuacion_actividad",
        "registro": f"{p}_ml_registro_modelos",
        "deriva": f"{p}_ml_monitoreo_deriva",
        "desempeno": f"{p}_ml_monitoreo_desempeno",
        "experimento": f"{p}_exp_actividad",
        "modelo": f"{p}_modelo_actividad",
    }


def validar_parametros_ml(
    contexto: Mapping[str, Any],
    p_lakehouse_esperado: str,
    p_prefijo: str,
    p_confirmar_reemplazo: str,
    exigir_confirmacion: bool = True,
) -> str:
    """Mismas protecciones que las sesiones 2 y 3: confirmacion explicita y Lakehouse correcto.

    Con `exigir_confirmacion=False` solo se validan el prefijo y el Lakehouse; sirve para
    explorar sin escribir. Toda escritura vuelve a llamarla con la confirmacion exigida.
    """
    if exigir_confirmacion and p_confirmar_reemplazo != "SI":
        raise ValueError(
            "p_confirmar_reemplazo must be exactly 'SI' to replace the prefixed ML tables"
        )
    if not isinstance(p_prefijo, str) or PATRON_PREFIJO.fullmatch(p_prefijo) is None:
        raise ValueError(
            "p_prefijo must start with an ASCII letter and contain at most "
            "20 ASCII letters, digits, or underscores"
        )
    if not isinstance(p_lakehouse_esperado, str):
        raise ValueError("p_lakehouse_esperado must be a string")
    if not isinstance(contexto, Mapping):
        raise ValueError("notebook runtime context must be a mapping")
    for clave in ("defaultLakehouseId", "defaultLakehouseWorkspaceId", "currentWorkspaceId"):
        valor = contexto.get(clave)
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError(f"notebook runtime context is missing {clave}")
    nombre = contexto.get("defaultLakehouseName")
    if not isinstance(nombre, str) or not nombre.strip():
        raise ValueError("notebook runtime context is missing defaultLakehouseName")
    if p_lakehouse_esperado and p_lakehouse_esperado != nombre:
        raise ValueError("p_lakehouse_esperado does not match the notebook default Lakehouse")
    if contexto["currentWorkspaceId"] != contexto["defaultLakehouseWorkspaceId"]:
        raise ValueError(
            "the default Lakehouse and current notebook must be in the same workspace"
        )
    return nombre


def _fecha_utc(ahora: datetime | None) -> str:
    momento = ahora if ahora is not None else datetime.now(timezone.utc)
    return momento.strftime("%Y-%m-%dT%H:%M:%SZ")


def leer_tabla(spark: Any, nombre: str, opcional: bool = False) -> pd.DataFrame | None:
    if not spark.catalog.tableExists(nombre):
        if opcional:
            return None
        raise ValueError(f"{nombre}: expected an existing table; observed missing")
    return spark.table(nombre).toPandas()


CONTEOS_SILVER = {"transacciones": 72000, "cuentas": 1800, "clientes": 1200}


def leer_silver(spark: Any, prefijo: str) -> dict[str, pd.DataFrame]:
    """Lee las tablas plata de la Sesion 3 y falla antes de cargar si no tienen el conteo esperado."""
    tablas: dict[str, pd.DataFrame] = {}
    for base, esperado in CONTEOS_SILVER.items():
        nombre = f"{prefijo.lower()}_silver_{base}"
        if not spark.catalog.tableExists(nombre):
            raise ValueError(f"{nombre}: expected an existing table; observed missing")
        observado = int(spark.table(nombre).count())
        if observado != esperado:
            raise ValueError(f"{nombre}: expected row count {esperado}; observed {observado}")
        tablas[base] = spark.table(nombre).toPandas()
    # Spark entrega decimal como objetos Decimal y el timestamp puede traer zona horaria.
    transacciones = tablas["transacciones"]
    fechas = pd.to_datetime(transacciones["fecha_hora"])
    if fechas.dt.tz is not None:
        fechas = fechas.dt.tz_localize(None)
    transacciones["fecha_hora"] = fechas
    transacciones["monto"] = _a_numerico(transacciones["monto"])
    tablas["cuentas"]["saldo_inicial"] = _a_numerico(tablas["cuentas"]["saldo_inicial"])
    return tablas


def escribir_tabla(
    spark: Any, nombre: str, df: pd.DataFrame, columnas: Sequence[tuple[str, str]]
) -> int:
    """Reemplaza la tabla Delta con tipos explicitos y comprueba el conteo escrito."""
    from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

    tipos = {"string": StringType(), "bigint": LongType(), "double": DoubleType()}
    nombres = [columna for columna, _ in columnas]
    faltantes = [columna for columna in nombres if columna not in df.columns]
    if faltantes:
        raise ValueError(f"{nombre}: expected columns {nombres!r}; observed missing {faltantes!r}")
    ordenado = df.loc[:, nombres].copy()
    for columna, tipo in columnas:
        if tipo == "double":
            ordenado[columna] = pd.to_numeric(ordenado[columna]).astype("float64")
        elif tipo == "bigint":
            serie = pd.to_numeric(ordenado[columna])
            if serie.isna().any():
                raise ValueError(f"{nombre}: column {columna}: expected no nulls in a bigint column")
            ordenado[columna] = serie.astype("int64")
    filas = a_filas(ordenado)
    esquema = StructType([StructField(c, tipos[t], True) for c, t in columnas])
    spark.createDataFrame(filas, schema=esquema).write.mode("overwrite").saveAsTable(nombre)
    observado = int(spark.table(nombre).count())
    if observado != len(filas):
        raise ValueError(f"{nombre}: expected table count {len(filas)}; observed {observado}")
    return observado


def reemplazar_corte(
    spark: Any,
    nombre: str,
    nuevo: pd.DataFrame,
    columnas: Sequence[tuple[str, str]],
    columna_corte: str,
    corte: str,
) -> int:
    """Idempotente: reemplaza solo las filas del corte y conserva las demas."""
    existente = leer_tabla(spark, nombre, opcional=True)
    if existente is not None and not existente.empty:
        existente = existente.loc[existente[columna_corte] != corte]
        nuevo = pd.concat([existente, nuevo], ignore_index=True)
    return escribir_tabla(spark, nombre, nuevo, columnas)


def cortes_para(corte_puntuacion: str, disponibles: Sequence[str]) -> dict[str, Any]:
    """Cortes de una ejecucion mensual: puntuar el corte, probar en el anterior, validar en el previo."""
    p = pd.Period(corte_puntuacion, freq="M")
    entrenamiento = tuple(sorted(c for c in disponibles if pd.Period(c, freq="M") < p - 2))
    if not entrenamiento:
        raise ValueError(
            f"cortes: expected at least one training cutoff before {p - 2}; observed none"
        )
    return {
        "entrenamiento": entrenamiento,
        "validacion": str(p - 2),
        "prueba": str(p - 1),
        "puntuacion": str(p),
    }


def campeon_actual(registro: pd.DataFrame | None) -> dict[str, Any] | None:
    """El campeon es el ultimo evento del registro (promocion o reversion)."""
    if registro is None or registro.empty:
        return None
    ultimo = registro.sort_values("secuencia").iloc[-1]
    return {
        "version": int(ultimo["version"]),
        "tipo_modelo": str(ultimo["tipo_modelo"]),
        "auc_validacion": float(ultimo["auc_validacion"]),
        "auc_prueba": float(ultimo["auc_prueba"]),
        "corte_prueba": str(ultimo["corte_prueba"]),
        "run_id": str(ultimo["run_id"]),
    }


def _nuevo_evento(
    registro: pd.DataFrame | None,
    evento: str,
    version: int,
    tipo_modelo: str,
    auc_validacion: float,
    auc_prueba: float,
    corte_prueba: str,
    run_id: str,
    motivo: str,
    ahora: datetime | None,
) -> pd.DataFrame:
    secuencia = 1 if registro is None or registro.empty else int(registro["secuencia"].max()) + 1
    fila = pd.DataFrame(
        [
            {
                "secuencia": secuencia,
                "fecha_utc": _fecha_utc(ahora),
                "evento": evento,
                "version": int(version),
                "tipo_modelo": tipo_modelo,
                "auc_validacion": float(auc_validacion),
                "auc_prueba": float(auc_prueba),
                "corte_prueba": corte_prueba,
                "run_id": run_id,
                "motivo": motivo,
            }
        ]
    )
    if registro is None or registro.empty:
        return fila
    return pd.concat([registro, fila], ignore_index=True)


def revertir_campeon(
    spark: Any, p_prefijo: str, version: int, motivo: str, ahora: datetime | None = None
) -> dict[str, Any]:
    """Reversion auditable: agrega un evento que vuelve a poner una version anterior como campeon."""
    if not motivo.strip():
        raise ValueError("motivo: expected a non-empty reason; observed an empty one")
    tabla = nombres_ml(p_prefijo)["registro"]
    registro = leer_tabla(spark, tabla)
    previos = registro.loc[registro["version"] == int(version)]
    if previos.empty:
        raise ValueError(
            f"version: expected one of {sorted(registro['version'].unique().tolist())!r}; "
            f"observed {version!r}"
        )
    base = previos.sort_values("secuencia").iloc[-1]
    actualizado = _nuevo_evento(
        registro, "reversion", int(version), str(base["tipo_modelo"]),
        float(base["auc_validacion"]), float(base["auc_prueba"]), str(base["corte_prueba"]),
        str(base["run_id"]), motivo, ahora,
    )
    escribir_tabla(spark, tabla, actualizado, COLUMNAS_REGISTRO)
    return campeon_actual(actualizado)  # type: ignore[return-value]


def _evaluar_en(modelo: Any, df: pd.DataFrame) -> dict[str, float]:
    probabilidad = modelo.predict_proba(df[list(FEATURES)])[:, 1]
    return evaluar_clasificador(df[ETIQUETA], probabilidad)


def ejecutar_ciclo(
    spark: Any,
    mlflow: Any,
    contexto: Mapping[str, Any],
    p_lakehouse_esperado: str,
    p_prefijo: str,
    p_confirmar_reemplazo: str,
    p_corte_puntuacion: str = CORTE_PUNTUACION,
    p_tipo_modelo: str = "logreg",
    p_factor_deriva: float = 0.0,
    ahora: datetime | None = None,
) -> dict[str, Any]:
    """Un ciclo mensual completo; se puede reejecutar sin duplicar filas ni versiones.

    1. Valida parametros y la tabla de features (compuerta de calidad de datos).
    2. Entrena un retador y lo compara con el campeon sobre el mismo corte de prueba.
    3. Promueve solo si cumple la regla; registra la version solo cuando se promueve.
    4. Puntua el corte con el campeon, mide la deriva y, si ya hay etiquetas del mes
       anterior, el desempeno real del modelo que puntuo ese mes.
    """
    lakehouse = validar_parametros_ml(
        contexto, p_lakehouse_esperado, p_prefijo, p_confirmar_reemplazo
    )
    nombres = nombres_ml(p_prefijo)
    fecha = _fecha_utc(ahora)

    features = leer_tabla(spark, nombres["features"])
    problemas = validar_features(features)
    if problemas:
        raise ValueError(f"{nombres['features']}: " + "; ".join(problemas))
    # Punto en el tiempo: al cierre del corte a puntuar no existen cortes posteriores ni se
    # conoce la etiqueta del propio corte. Asi una ejecucion retrospectiva no ve el futuro.
    visibles = features.loc[features[CORTE] <= p_corte_puntuacion].copy()
    visibles.loc[visibles[CORTE] == p_corte_puntuacion, ETIQUETA] = np.nan
    cortes = cortes_para(p_corte_puntuacion, sorted(visibles[CORTE].unique()))
    partes = dividir_temporal(
        visibles,
        entrenamiento=cortes["entrenamiento"],
        validacion=cortes["validacion"],
        prueba=cortes["prueba"],
        puntuacion=cortes["puntuacion"],
    )

    registro = leer_tabla(spark, nombres["registro"], opcional=True)
    campeon = campeon_actual(registro)

    entrenamiento, validacion, prueba = (
        partes["entrenamiento"], partes["validacion"], partes["prueba"]
    )
    mlflow.set_experiment(nombres["experimento"])
    version: int | None = None
    with mlflow.start_run(run_name=f"retador_{p_tipo_modelo}_{p_corte_puntuacion}") as corrida:
        run_id = corrida.info.run_id
        retador = crear_modelo(p_tipo_modelo).fit(
            entrenamiento[list(FEATURES)], entrenamiento[ETIQUETA].astype(int)
        )
        metricas_val = _evaluar_en(retador, validacion)
        metricas_prueba = _evaluar_en(retador, prueba)
        mlflow.log_params(
            {
                "tipo_modelo": p_tipo_modelo,
                "cortes_entrenamiento": ",".join(cortes["entrenamiento"]),
                "corte_validacion": cortes["validacion"],
                "corte_prueba": cortes["prueba"],
                "umbral_alta_actividad": UMBRAL_ALTA_ACTIVIDAD,
                "filas_entrenamiento": len(entrenamiento),
            }
        )
        mlflow.log_metrics(
            {
                **{f"{k}_validacion": v for k, v in metricas_val.items()},
                **{f"{k}_prueba": v for k, v in metricas_prueba.items()},
            }
        )

        metricas_campeon = None
        if campeon is not None:
            modelo_campeon = mlflow.sklearn.load_model(
                f"models:/{nombres['modelo']}/{campeon['version']}"
            )
            metricas_campeon = _evaluar_en(modelo_campeon, prueba)
            mlflow.log_metric("auc_prueba_campeon", metricas_campeon["auc"])
        decision = decidir_promocion(metricas_prueba, metricas_campeon)
        mlflow.set_tags(
            {"decision": decision["decision"], "corte_puntuacion": p_corte_puntuacion}
        )

        firma = mlflow.models.infer_signature(
            entrenamiento[list(FEATURES)],
            pd.Series(retador.predict(entrenamiento[list(FEATURES)]), name="prediccion"),
        )
        mlflow.sklearn.log_model(
            retador, "model", signature=firma, serialization_format="cloudpickle"
        )
        if decision["decision"] == "promover":
            registrada = mlflow.register_model(
                f"runs:/{run_id}/model",
                nombres["modelo"],
                tags={
                    "tipo_modelo": p_tipo_modelo,
                    "corte_prueba": cortes["prueba"],
                    "auc_prueba": f"{metricas_prueba['auc']:.4f}",
                    "umbral_alta_actividad": str(UMBRAL_ALTA_ACTIVIDAD),
                },
            )
            version = int(registrada.version)

    if version is not None:
        registro = _nuevo_evento(
            registro, "promocion", version, p_tipo_modelo, metricas_val["auc"],
            metricas_prueba["auc"], cortes["prueba"], run_id, decision["motivo"], ahora,
        )
        escribir_tabla(spark, nombres["registro"], registro, COLUMNAS_REGISTRO)
        campeon = campeon_actual(registro)
    if campeon is None:
        raise ValueError(
            f"{nombres['modelo']}: expected a champion model; observed none "
            f"(the challenger was rejected: {decision['motivo']})"
        )

    modelo_activo = mlflow.sklearn.load_model(f"models:/{nombres['modelo']}/{campeon['version']}")
    por_puntuar = partes["puntuacion"]
    if p_factor_deriva and p_factor_deriva > 0:
        por_puntuar = simular_deriva_actividad(por_puntuar, p_factor_deriva)
    puntuacion = puntuar(modelo_activo, por_puntuar)
    puntuacion["version_modelo"] = campeon["version"]
    puntuacion["fecha_utc"] = fecha
    reemplazar_corte(
        spark, nombres["puntuacion"], puntuacion, COLUMNAS_PUNTUACION, CORTE,
        cortes["puntuacion"],
    )

    deriva = reporte_deriva(entrenamiento, por_puntuar)
    deriva_tabla = deriva.assign(
        corte=cortes["puntuacion"], simulada=int(bool(p_factor_deriva)), fecha_utc=fecha
    )
    reemplazar_corte(
        spark, nombres["deriva"], deriva_tabla, COLUMNAS_DERIVA, "corte", cortes["puntuacion"]
    )

    auc_real = None
    anterior = str(pd.Period(p_corte_puntuacion, freq="M") - 1)
    previas = leer_tabla(spark, nombres["puntuacion"], opcional=True)
    if previas is not None:
        previas = previas.loc[previas[CORTE] == anterior]
        reales = features.loc[(features[CORTE] == anterior) & features[ETIQUETA].notna()]
        union = previas.merge(reales[[CLAVE, ETIQUETA]], on=CLAVE, how="inner")
        if not union.empty:
            metricas_reales = evaluar_clasificador(
                union[ETIQUETA], union["probabilidad_alta_actividad"]
            )
            auc_real = metricas_reales["auc"]
            medicion = pd.DataFrame(
                [
                    {
                        "corte_prediccion": anterior,
                        "mes_evaluado": str(pd.Period(anterior, freq="M") + 1),
                        "version_modelo": int(union["version_modelo"].iloc[0]),
                        "filas": len(union),
                        "auc": auc_real,
                        "average_precision": metricas_reales["average_precision"],
                        "prevalencia_real": metricas_reales["prevalencia"],
                        "fecha_utc": fecha,
                    }
                ]
            )
            reemplazar_corte(
                spark, nombres["desempeno"], medicion, COLUMNAS_DESEMPENO,
                "corte_prediccion", anterior,
            )

    salud = evaluar_alertas(deriva, auc_actual=auc_real, auc_referencia=campeon["auc_prueba"])
    informe = {
        "status": "success",
        "lakehouse": lakehouse,
        "corte_puntuacion": cortes["puntuacion"],
        "cortes": {k: list(v) if isinstance(v, tuple) else v for k, v in cortes.items()},
        "decision": decision["decision"],
        "motivo": decision["motivo"],
        "version_campeon": campeon["version"],
        "retador": {k: round(v, 6) for k, v in metricas_prueba.items()},
        "campeon_previo": (
            None if metricas_campeon is None
            else {k: round(v, 6) for k, v in metricas_campeon.items()}
        ),
        "deriva": {k: int(v) for k, v in deriva["deriva"].value_counts().items()},
        "deriva_simulada": bool(p_factor_deriva),
        "auc_real_mes_anterior": None if auc_real is None else round(auc_real, 6),
        "alertas": salud["alertas"],
        "recomendacion": salud["recomendacion"],
    }
    print(json.dumps(informe, ensure_ascii=False, sort_keys=True))
    return informe


In [ ]:
"""Caso de ML del bonus B6: deteccion de transacciones atipicas con puntuacion en linea.

Pregunta: dada una transaccion en el momento en que ocurre, ¿se parece a lo que esa
cuenta suele hacer? No hay etiquetas de fraude: se entrena un modelo no supervisado
(IsolationForest) solo con comportamiento normal y se mide con **anomalias inyectadas**
en los meses de validacion y prueba, cuya naturaleza y cantidad se conocen.

Los datos son sinteticos: las metricas sirven para practicar el ciclo de vida, no para
estimar el rendimiento de un detector real. El modulo solo usa pandas, numpy y
scikit-learn, que forman parte del runtime de Fabric, y no depende de `ml_actividad.py`.
Las variables se calculan con **una sola funcion** (`construir_features_tx`) tanto para
entrenar como para puntuar una transaccion en linea: esa es la forma de evitar que el
entrenamiento y el servicio calculen cosas distintas.
"""

from __future__ import annotations

import math
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd


CLAVE_TX = "transaccion_id"
CUENTA = "cuenta_id"
FECHA = "fecha_hora"
MES = "mes"
ETIQUETA_INYECTADA = "es_anomalia_inyectada"
TIPO_INYECTADO = "tipo_anomalia"

# Los meses de entrenamiento se usan tal como estan (supuesto: casi todo es normal).
# Abril solo aporta historial: una transaccion necesita varias previas de su cuenta.
MESES_ENTRENAMIENTO = ("2026-05", "2026-06", "2026-07")
MES_VALIDACION = "2026-08"
MES_PRUEBA = "2026-09"
MINIMO_HISTORIAL = 10
VENTANA_CANAL = 10
TASA_REVISION = 0.01
ENTRENAMIENTO_MAXIMO = 20000
TASA_INYECCION = 0.01
SEMILLA_VALIDACION = 11
SEMILLA_PRUEBA = 23

TIPOS_ANOMALIA = ("monto_extremo", "madrugada", "rafaga")
MAXIMO_MINUTOS_SIN_ACTIVIDAD = 720.0 * 60.0
PISO_MAD = 0.05

FEATURES_NUMERICAS_TX = (
    "z_monto_tipo",
    "hora",
    "es_fin_semana",
    "log_minutos_desde_ultima",
    "tx_ultimas_24h",
    "pct_canal_historial",
)
FEATURES_CATEGORICAS_TX = ("canal", "tipo_transaccion")
FEATURES_TX = FEATURES_NUMERICAS_TX + FEATURES_CATEGORICAS_TX
# Columnas enteras: en la solicitud al endpoint deben viajar como enteros, no como decimales.
FEATURES_ENTERAS_TX = ("hora", "es_fin_semana", "tx_ultimas_24h")

_COLUMNAS_TX = (CLAVE_TX, CUENTA, FECHA, "tipo_transaccion", "direccion", "canal", "monto")

COLUMNAS_REFERENCIA_TX = (
    ("tipo_transaccion", "string"),
    ("mediana_log_monto", "double"),
    ("mad_log_monto", "double"),
    ("max_monto", "double"),
    ("filas", "bigint"),
    ("fecha_utc", "string"),
)
COLUMNAS_ANOMALIAS_TX = (
    (CLAVE_TX, "string"),
    (CUENTA, "string"),
    (MES, "string"),
    (FECHA, "string"),
    ("tipo_transaccion", "string"),
    ("canal", "string"),
    ("monto", "double"),
    ("score_anomalia", "double"),
    ("es_atipica", "bigint"),
    (ETIQUETA_INYECTADA, "bigint"),
    (TIPO_INYECTADO, "string"),
    ("version_modelo", "bigint"),
    ("fecha_utc", "string"),
)


def nombres_anomalias(prefijo: str) -> dict[str, str]:
    p = prefijo.lower()
    return {
        "referencia": f"{p}_gold_ml_referencia_tx",
        "anomalias": f"{p}_gold_ml_anomalias_tx",
        "experimento": f"{p}_exp_atipicas_tx",
        "modelo": f"{p}_modelo_atipicas_tx",
    }


def _exigir_columnas_tx(df: pd.DataFrame, columnas: Sequence[str], nombre: str) -> None:
    faltantes = [columna for columna in columnas if columna not in df.columns]
    if faltantes:
        raise ValueError(
            f"{nombre}: expected columns {list(columnas)!r}; observed missing {faltantes!r}"
        )


def preparar_transacciones(transacciones: pd.DataFrame) -> pd.DataFrame:
    """Copia ordenada, con fecha sin zona y monto decimal, lista para calcular variables."""
    _exigir_columnas_tx(transacciones, _COLUMNAS_TX, "transacciones")
    tx = transacciones.loc[:, list(_COLUMNAS_TX)].copy()
    tx[FECHA] = pd.to_datetime(tx[FECHA])
    if tx[FECHA].dt.tz is not None:
        tx[FECHA] = tx[FECHA].dt.tz_localize(None)
    tx[FECHA] = tx[FECHA].astype("datetime64[ns]")
    tx["monto"] = pd.to_numeric(tx["monto"].astype(float), errors="raise")
    if (tx["monto"] <= 0).any():
        raise ValueError("transacciones: expected positive amounts; observed non-positive values")
    if tx[CLAVE_TX].duplicated().any():
        raise ValueError(f"transacciones: expected unique {CLAVE_TX}; observed duplicates")
    return tx.sort_values([CUENTA, FECHA, CLAVE_TX], kind="mergesort").reset_index(drop=True)


def calcular_referencia(transacciones: pd.DataFrame, fecha_utc: str = "") -> pd.DataFrame:
    """Mediana y desviacion robusta (MAD) del logaritmo del monto por tipo, y el monto maximo.

    Se calcula **solo con los meses de entrenamiento**. Es la referencia que el servicio
    en linea necesita para calcular `z_monto_tipo` de una transaccion nueva.
    """
    tx = preparar_transacciones(transacciones)
    tx["log_monto"] = np.log(tx["monto"])
    filas = []
    for tipo, grupo in tx.groupby("tipo_transaccion"):
        mediana = float(grupo["log_monto"].median())
        mad = float((grupo["log_monto"] - mediana).abs().median() * 1.4826)
        filas.append(
            {
                "tipo_transaccion": str(tipo),
                "mediana_log_monto": mediana,
                "mad_log_monto": max(mad, PISO_MAD),
                "max_monto": float(grupo["monto"].max()),
                "filas": int(len(grupo)),
                "fecha_utc": fecha_utc,
            }
        )
    return pd.DataFrame(filas).sort_values("tipo_transaccion").reset_index(drop=True)


def construir_features_tx(
    transacciones: pd.DataFrame,
    referencia: pd.DataFrame,
    minimo_historial: int = MINIMO_HISTORIAL,
) -> pd.DataFrame:
    """Una fila por transaccion con variables que solo usan lo ocurrido **antes** de ella.

    `transacciones` debe traer todo el historial necesario (no solo el mes a puntuar).
    Se descartan las transacciones con menos de `minimo_historial` previas en su cuenta.
    Las columnas extra (por ejemplo, las etiquetas inyectadas) se conservan.
    """
    base = preparar_transacciones(transacciones)
    extras = [c for c in transacciones.columns if c not in _COLUMNAS_TX]
    if extras:
        base = base.merge(transacciones[[CLAVE_TX, *extras]], on=CLAVE_TX, how="left")
        base = base.sort_values([CUENTA, FECHA, CLAVE_TX], kind="mergesort").reset_index(drop=True)
    _exigir_columnas_tx(referencia, ("tipo_transaccion", "mediana_log_monto", "mad_log_monto"), "referencia")
    desconocidos = sorted(set(base["tipo_transaccion"]) - set(referencia["tipo_transaccion"]))
    if desconocidos:
        raise ValueError(
            f"referencia: expected every transaction type; observed missing {desconocidos!r}"
        )
    ref = referencia.set_index("tipo_transaccion")

    grupo = base.groupby(CUENTA, sort=False)
    base["n_previas"] = grupo.cumcount()
    # Parte de las ultimas VENTANA_CANAL transacciones previas de la cuenta que uso este mismo canal.
    # Una ventana de tamano fijo evita que la variable cambie solo porque el historial crece.
    misma = pd.Series(0.0, index=base.index)
    for canal in base["canal"].unique():
        es_canal = (base["canal"] == canal).astype(float)
        previas_canal = es_canal.groupby(base[CUENTA], sort=False).transform(
            lambda s: s.shift(1).rolling(VENTANA_CANAL, min_periods=1).sum()
        )
        misma = misma.where(base["canal"] != canal, previas_canal.fillna(0.0))
    base["pct_canal_historial"] = misma / base["n_previas"].clip(lower=1, upper=VENTANA_CANAL)
    segundos = base[FECHA].astype("datetime64[ns]").astype("int64") / 1e9
    base["_segundos"] = segundos
    anterior = base.groupby(CUENTA, sort=False)["_segundos"].shift(1)
    minutos = ((base["_segundos"] - anterior) / 60.0).clip(upper=MAXIMO_MINUTOS_SIN_ACTIVIDAD)
    base["log_minutos_desde_ultima"] = np.log1p(minutos)

    def _ultimas_24h(serie: pd.Series) -> np.ndarray:
        s = serie.to_numpy()
        return np.arange(len(s)) - np.searchsorted(s, s - 24 * 3600, side="left")

    base["tx_ultimas_24h"] = base.groupby(CUENTA, sort=False)["_segundos"].transform(_ultimas_24h)

    mediana = base["tipo_transaccion"].map(ref["mediana_log_monto"])
    mad = base["tipo_transaccion"].map(ref["mad_log_monto"])
    base["z_monto_tipo"] = (np.log(base["monto"]) - mediana) / mad
    base["hora"] = base[FECHA].dt.hour.astype("int64")
    base["es_fin_semana"] = (base[FECHA].dt.day_of_week >= 5).astype("int64")
    base["tx_ultimas_24h"] = base["tx_ultimas_24h"].astype("int64")
    base[MES] = base[FECHA].dt.strftime("%Y-%m")

    elegibles = base.loc[base["n_previas"] >= minimo_historial].copy()
    columnas = [CLAVE_TX, CUENTA, MES, FECHA, "monto", "n_previas", *FEATURES_TX, *extras]
    return elegibles.loc[:, columnas].reset_index(drop=True)


def inyectar_anomalias(
    transacciones: pd.DataFrame,
    semillas_por_mes: Mapping[str, int],
    tasa: float = TASA_INYECCION,
    minimo_historial: int = MINIMO_HISTORIAL,
) -> pd.DataFrame:
    """Copia de `transacciones` con anomalias conocidas en cada mes de `semillas_por_mes` (mes -> semilla).

    Tipos (se reparten por igual; `tasa` es la fraccion de filas del mes que queda etiquetada):

    * `monto_extremo`: el monto se multiplica por un factor entre 3 y 8.
    * `madrugada`: la hora pasa a la madrugada (0 a 5 h, que en estos datos nunca ocurre)
      y el monto se multiplica por un factor entre 1 y 2,5.
    * `rafaga`: tras la transaccion elegida se agregan tres mas de la misma cuenta, tipo y canal,
      en los siguientes 20 minutos y con montos parecidos. Las tres agregadas son las etiquetadas.
    """
    if not 0 < tasa < 0.2:
        raise ValueError(f"tasa: expected a number in (0, 0.2); observed {tasa!r}")
    tx = preparar_transacciones(transacciones)
    tx[ETIQUETA_INYECTADA] = np.int64(0)
    tx[TIPO_INYECTADO] = ""
    nuevas: list[dict[str, Any]] = []
    contador = 0

    previas = tx.groupby(CUENTA, sort=False).cumcount()
    mes_fila = tx[FECHA].dt.strftime("%Y-%m")
    # Margen extra: mover una transaccion a la madrugada la adelanta y puede quitarle historial.
    margen_historial = minimo_historial + 3
    for mes, semilla in semillas_por_mes.items():
        rng = np.random.default_rng(semilla)
        candidatas = np.flatnonzero(
            ((mes_fila == mes) & (previas >= margen_historial) & (tx[FECHA].dt.day <= 27)).to_numpy()
        )
        por_tipo = int(round(tasa * int((mes_fila == mes).sum()) / 5))
        if por_tipo * 3 > len(candidatas):
            raise ValueError(
                f"{mes}: expected at least {por_tipo * 3} eligible rows; observed {len(candidatas)}"
            )
        elegidas = rng.choice(candidatas, size=por_tipo * 3, replace=False)
        for tipo, indices in zip(TIPOS_ANOMALIA, np.split(elegidas, 3)):
            for i in indices:
                fila = tx.iloc[i]
                if tipo == "monto_extremo":
                    tx.at[i, "monto"] = round(float(fila["monto"]) * float(rng.uniform(3, 8)), 2)
                    tx.at[i, ETIQUETA_INYECTADA], tx.at[i, TIPO_INYECTADO] = 1, tipo
                elif tipo == "madrugada":
                    momento = fila[FECHA].normalize() + pd.Timedelta(
                        hours=int(rng.integers(0, 6)),
                        minutes=int(rng.integers(0, 60)),
                        seconds=int(rng.integers(0, 60)),
                    )
                    tx.at[i, FECHA] = momento
                    tx.at[i, "monto"] = round(float(fila["monto"]) * float(rng.uniform(1, 2.5)), 2)
                    tx.at[i, ETIQUETA_INYECTADA], tx.at[i, TIPO_INYECTADO] = 1, tipo
                else:
                    momento = fila[FECHA]
                    for _ in range(3):
                        contador += 1
                        momento = momento + pd.Timedelta(
                            minutes=float(rng.uniform(1, 6)), seconds=int(rng.integers(0, 60))
                        )
                        nuevas.append(
                            {
                                CLAVE_TX: f"TXR{contador:06d}",
                                CUENTA: fila[CUENTA],
                                FECHA: momento,
                                "tipo_transaccion": fila["tipo_transaccion"],
                                "direccion": fila["direccion"],
                                "canal": fila["canal"],
                                "monto": round(float(fila["monto"]) * float(rng.uniform(0.9, 1.1)), 2),
                                ETIQUETA_INYECTADA: np.int64(1),
                                TIPO_INYECTADO: tipo,
                            }
                        )
    if nuevas:
        tx = pd.concat([tx, pd.DataFrame(nuevas)], ignore_index=True)
    tx[ETIQUETA_INYECTADA] = tx[ETIQUETA_INYECTADA].astype("int64")
    return tx.sort_values([CUENTA, FECHA, CLAVE_TX], kind="mergesort").reset_index(drop=True)


def muestrear_entrenamiento(df: pd.DataFrame, maximo: int = ENTRENAMIENTO_MAXIMO, semilla: int = 42) -> pd.DataFrame:
    """Muestra aleatoria reproducible: LOF y OneClassSVM guardan o comparan contra los datos de entrenamiento."""
    if maximo < 1:
        raise ValueError(f"maximo: expected a positive integer; observed {maximo!r}")
    if len(df) <= maximo:
        return df.reset_index(drop=True)
    return df.sample(n=maximo, random_state=semilla).sort_index().reset_index(drop=True)


def crear_detector(tipo: str = "lof", vecinos: int = 100, semilla: int = 42) -> Any:
    """Pipeline de scikit-learn: escalado y one-hot + un detector no supervisado.

    `tipo`: `iforest` (IsolationForest) o `lof` (LocalOutlierFactor con `novelty=True`, que permite
    puntuar datos nuevos). Los dos exponen `offset_`, el umbral que usa `predict`.
    """
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import IsolationForest
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    detectores = {
        "iforest": lambda: IsolationForest(n_estimators=200, random_state=semilla, n_jobs=1),
        "lof": lambda: LocalOutlierFactor(n_neighbors=vecinos, novelty=True),
    }
    if tipo not in detectores:
        raise ValueError(f"tipo: expected one of {sorted(detectores)!r}; observed {tipo!r}")
    preparacion = ColumnTransformer(
        [
            ("numericas", StandardScaler(), list(FEATURES_NUMERICAS_TX)),
            ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(FEATURES_CATEGORICAS_TX)),
        ]
    )
    return Pipeline([("preparacion", preparacion), ("detector", detectores[tipo]())])


def _puntaje_crudo(modelo: Any, df: pd.DataFrame) -> np.ndarray:
    """`score_samples` de scikit-learn: cuanto mas bajo, mas atipica la transaccion."""
    transformadas = modelo.named_steps["preparacion"].transform(df[list(FEATURES_TX)])
    return modelo.named_steps["detector"].score_samples(transformadas)


def fijar_umbral(modelo: Any, validacion: pd.DataFrame, tasa: float = TASA_REVISION) -> float:
    """Ajusta el umbral para marcar aproximadamente `tasa` de la validacion y lo guarda en el modelo.

    `predict` marca como atipica la transaccion cuyo puntaje es menor que `offset_`. Fijar
    `offset_` deja el umbral **dentro del modelo registrado**: el lote, `PREDICT` y el endpoint
    usan exactamente el mismo. La `tasa` es la capacidad de revision (cuantas se pueden revisar).
    """
    if not 0 < tasa < 0.2:
        raise ValueError(f"tasa: expected a number in (0, 0.2); observed {tasa!r}")
    umbral = float(np.quantile(_puntaje_crudo(modelo, validacion), tasa))
    modelo.named_steps["detector"].offset_ = umbral
    return umbral

def puntuar_anomalias(modelo: Any, df: pd.DataFrame) -> pd.DataFrame:
    """Puntaje de anomalia (mayor = mas atipica) y marca de atipica (1) segun el umbral del modelo."""
    puntaje = -modelo.decision_function(df[list(FEATURES_TX)])
    marca = (modelo.predict(df[list(FEATURES_TX)]) == -1).astype("int64")
    return pd.DataFrame(
        {
            CLAVE_TX: df[CLAVE_TX].to_numpy(),
            "score_anomalia": np.asarray(puntaje, dtype=float),
            "es_atipica": marca,
        }
    )


def evaluar_deteccion(y_real: Any, puntaje: Any, marcadas: Any) -> dict[str, float]:
    from sklearn.metrics import average_precision_score, roc_auc_score

    y = np.asarray(y_real, dtype=int)
    p = np.asarray(puntaje, dtype=float)
    m = np.asarray(marcadas, dtype=int)
    positivos_marcados = int(((y == 1) & (m == 1)).sum())
    return {
        "auc": float(roc_auc_score(y, p)),
        "average_precision": float(average_precision_score(y, p)),
        "precision": positivos_marcados / max(int(m.sum()), 1),
        "recall": positivos_marcados / max(int(y.sum()), 1),
        "tasa_marcadas": float(m.mean()),
        "prevalencia": float(y.mean()),
    }


def regla_maximo_por_tipo(df: pd.DataFrame, referencia: pd.DataFrame) -> pd.DataFrame:
    """Linea base sin aprendizaje: marca una transaccion si supera el monto maximo de su tipo en entrenamiento."""
    maximo = df["tipo_transaccion"].map(referencia.set_index("tipo_transaccion")["max_monto"])
    razon = df["monto"].to_numpy() / maximo.to_numpy()
    return pd.DataFrame(
        {CLAVE_TX: df[CLAVE_TX].to_numpy(), "score_anomalia": razon, "es_atipica": (razon > 1.0).astype("int64")}
    )


def recall_por_tipo(df: pd.DataFrame, marcadas: Any) -> pd.DataFrame:
    """Proporcion de anomalias inyectadas de cada tipo que quedaron marcadas."""
    marcado = pd.Series(np.asarray(marcadas, dtype=int), index=df.index)
    inyectadas = df.loc[df[ETIQUETA_INYECTADA] == 1]
    tabla = (
        pd.DataFrame({"tipo_anomalia": inyectadas[TIPO_INYECTADO], "marcada": marcado.loc[inyectadas.index]})
        .groupby("tipo_anomalia")["marcada"]
        .agg(inyectadas="size", marcadas="sum")
        .reindex(TIPOS_ANOMALIA)
        .fillna(0)
        .astype("int64")
    )
    tabla["recall"] = tabla["marcadas"] / tabla["inyectadas"].clip(lower=1)
    return tabla.reset_index()


def _nativo(valor: Any) -> Any:
    if isinstance(valor, np.generic):
        valor = valor.item()
    if isinstance(valor, float) and math.isnan(valor):
        raise ValueError("solicitud: expected no missing values; observed NaN")
    return valor


def solicitud_endpoint(df: pd.DataFrame) -> dict[str, Any]:
    """Cuerpo JSON para `.../endpoint/versions/<n>/score` con las columnas en el orden de la firma."""
    filas = []
    for _, fila in df[list(FEATURES_TX)].iterrows():
        valores = []
        for columna in FEATURES_TX:
            valor = _nativo(fila[columna])
            if columna in FEATURES_ENTERAS_TX:
                valor = int(valor)
            elif columna in FEATURES_CATEGORICAS_TX:
                valor = str(valor)
            else:
                valor = float(valor)
            valores.append(valor)
        filas.append(valores)
    return {"formatType": "dataframe", "orientation": "values", "inputs": filas}


def features_de_una_transaccion(
    historial: pd.DataFrame, nueva: Mapping[str, Any], referencia: pd.DataFrame
) -> pd.DataFrame:
    """Variables de una transaccion nueva, calculadas con el **mismo codigo** que el entrenamiento.

    `historial` son las transacciones previas de la cuenta (o de varias). La nueva se agrega al
    final; no se necesita nada del futuro.
    """
    _exigir_columnas_tx(pd.DataFrame([nueva]), _COLUMNAS_TX, "nueva")
    cuenta = nueva[CUENTA]
    previo = historial.loc[historial[CUENTA] == cuenta, list(_COLUMNAS_TX)]
    posterior = previo.loc[pd.to_datetime(previo[FECHA]) > pd.to_datetime(nueva[FECHA])]
    if not posterior.empty:
        raise ValueError("historial: expected only transactions before the new one; observed later ones")
    unido = pd.concat([previo, pd.DataFrame([dict(nueva)])[list(_COLUMNAS_TX)]], ignore_index=True)
    resultado = construir_features_tx(unido, referencia)
    fila = resultado.loc[resultado[CLAVE_TX] == nueva[CLAVE_TX]]
    if fila.empty:
        raise ValueError(
            f"historial: expected at least {MINIMO_HISTORIAL} previous transactions for {cuenta}; observed fewer"
        )
    return fila.reset_index(drop=True)


def elegir_candidato(candidatos: pd.DataFrame, margen: float = 0.01) -> str:
    """Entre los candidatos con AP de validacion a menos de `margen` del mejor, el menos complejo.

    `candidatos` trae `nombre`, `average_precision_validacion` y `complejidad` (cuanto menor, mas simple).
    """
    _exigir_columnas_tx(candidatos, ("nombre", "average_precision_validacion", "complejidad"), "candidatos")
    if candidatos.empty:
        raise ValueError("candidatos: expected at least one row; observed none")
    mejor = float(candidatos["average_precision_validacion"].max())
    cercanos = candidatos.loc[candidatos["average_precision_validacion"] >= mejor - margen]
    return str(cercanos.sort_values(["complejidad", "nombre"]).iloc[0]["nombre"])


def diferencias_entre_features(completas: pd.DataFrame, recortadas: pd.DataFrame) -> int:
    """Cuenta valores distintos en las filas comunes: debe ser 0 si no hay fuga de datos.

    `recortadas` se calcula con los datos cortados en una fecha; las variables de las transacciones
    anteriores a esa fecha no pueden cambiar.
    """
    comunes = completas.merge(recortadas, on=CLAVE_TX, suffixes=("", "_recortada"))
    if comunes.empty:
        raise ValueError("features: expected rows in common; observed none")
    distintos = 0
    for columna in FEATURES_TX:
        if columna in FEATURES_CATEGORICAS_TX:
            distintos += int((comunes[columna] != comunes[columna + "_recortada"]).sum())
        else:
            distintos += int((~np.isclose(comunes[columna], comunes[columna + "_recortada"])).sum())
    return distintos


def puntuar_en_linea(
    cliente: Any, id_workspace: str, id_modelo: str, cuerpo: Mapping[str, Any], version: int | None = None
) -> dict[str, Any]:
    """Llama al endpoint en tiempo real de una version activa; nunca activa ninguna por su cuenta.

    Sin `version`, usa la mas reciente de las versiones activas. `cliente` es un cliente REST de Fabric
    (`sempy.fabric.FabricRestClient`) o algo con la misma forma: `get(ruta)` y `post(ruta, json=...)`
    que devuelven un objeto con `status_code` y `json()`.
    """
    ruta = f"/v1/workspaces/{id_workspace}/mlmodels/{id_modelo}/endpoint/versions"
    respuesta = cliente.get(ruta)
    if respuesta.status_code != 200:
        return {"estado": f"consulta fallida (HTTP {respuesta.status_code})", "version": version, "predicciones": None}
    estados = {str(v.get("versionName")): v.get("status") for v in respuesta.json().get("value", [])}
    if version is None:
        activas = sorted(int(n) for n, estado in estados.items() if estado == "Active")
        if not activas:
            resumen = ", ".join(f"v{n}: {e}" for n, e in sorted(estados.items())) or "sin versiones"
            return {"estado": f"ninguna version activa ({resumen})", "version": None, "predicciones": None}
        version = activas[-1]
    estado = estados.get(str(version), "sin endpoint")
    if estado != "Active":
        return {"estado": estado, "version": version, "predicciones": None}
    from time import perf_counter

    inicio = perf_counter()
    resultado = cliente.post(f"{ruta}/{version}/score", json=dict(cuerpo))
    segundos = perf_counter() - inicio
    if resultado.status_code != 200:
        return {"estado": f"puntuacion fallida (HTTP {resultado.status_code})", "version": version, "predicciones": None}
    predicciones = [int(fila[0]) for fila in resultado.json()["predictions"]]
    return {"estado": estado, "version": version, "predicciones": predicciones, "segundos": segundos}


## 1. Preparar

Se reutilizan las protecciones de B1 a B4 (Lakehouse correcto, prefijo válido) y las tablas plata de la Sesión 3. `p_tasa_revision` es la **capacidad de revisión**: la fracción de transacciones que el equipo puede revisar a mano (por defecto 1 %).

`mlflow.autolog(disable=True)` desactiva el registro automático: en Fabric viene activo y cada `fit` crearía una corrida extra con nombre automático. Aquí se registra **exactamente** lo que se decide registrar (igual que en B4).


In [ ]:
import notebookutils
import mlflow
from mlflow.models import infer_signature

mlflow.autolog(disable=True)
lakehouse = validar_parametros_ml(
    notebookutils.runtime.context, p_lakehouse_esperado, p_prefijo, p_confirmar_reemplazo,
    exigir_confirmacion=False,
)
nombres = nombres_anomalias(p_prefijo)
tasa_revision = float(p_tasa_revision)
silver = leer_silver(spark, p_prefijo)
tx = preparar_transacciones(silver["transacciones"])
mes_de = tx[FECHA].dt.strftime("%Y-%m")
transacciones_entrenamiento = tx.loc[mes_de.isin(MESES_ENTRENAMIENTO)]
print({"transacciones": len(tx), "meses_entrenamiento": list(MESES_ENTRENAMIENTO), "filas_entrenamiento": len(transacciones_entrenamiento)})


## 2. La referencia y las variables de cada transacción

**Referencia.** Para saber si un monto es alto hay que compararlo con lo normal **de su tipo** (500 000 es poco para un salario y mucho para una compra). La referencia guarda la mediana y la desviación robusta del logaritmo del monto por tipo, calculadas **solo con los meses de entrenamiento**.

**Variables.** Cada una usa únicamente lo que ocurrió **antes** de la transacción:

| Variable | Qué mide |
|---|---|
| `z_monto_tipo` | Cuántas desviaciones robustas se aleja el monto de lo normal de su tipo |
| `hora`, `es_fin_semana` | Cuándo ocurrió |
| `log_minutos_desde_ultima` | Tiempo desde la transacción anterior de la cuenta, en escala logarítmica (los minutos y las horas pesan distinto) |
| `tx_ultimas_24h` | Cuántas transacciones previas tuvo la cuenta en las últimas 24 horas |
| `pct_canal_historial` | Qué parte del historial de la cuenta usó este mismo canal (bajo = canal inusual para ella) |
| `canal`, `tipo_transaccion` | Categóricas |

Una transacción necesita al menos tres previas en su cuenta para tener historial; abril solo aporta historial.


In [ ]:
referencia = calcular_referencia(transacciones_entrenamiento)
display(referencia.round(3))


## 3. Anomalías conocidas para poder medir

En agosto (validación) y septiembre (prueba) se **inyectan** anomalías, cerca del 1 % de las filas de cada mes, repartidas por igual en tres tipos:

| Tipo | Qué se hace |
|---|---|
| `monto_extremo` | El monto se multiplica por un factor entre 3 y 8 |
| `madrugada` | La hora pasa a las 0 a 5 h (en estos datos **nunca** ocurre) y el monto sube entre 1 y 2,5 veces |
| `rafaga` | Tras una transacción se agregan tres más de la misma cuenta, tipo y canal en menos de 20 minutos |

Cada fila lleva su etiqueta (`es_anomalia_inyectada`) y su tipo. Se usan **semillas distintas** para validación y prueba.


In [ ]:
tx_evaluacion = inyectar_anomalias(tx, {MES_VALIDACION: SEMILLA_VALIDACION, MES_PRUEBA: SEMILLA_PRUEBA})
features = construir_features_tx(tx_evaluacion, referencia)
resumen_inyeccion = (
    features.loc[features[ETIQUETA_INYECTADA] == 1]
    .groupby([MES, TIPO_INYECTADO]).size().rename("filas").reset_index()
)
display(resumen_inyeccion)
print({"filas_con_variables": len(features), "inyectadas": int(features[ETIQUETA_INYECTADA].sum())})


### Prueba de fuga de datos

Si se **cortan los datos a mitad de septiembre**, las variables de todas las transacciones anteriores al corte no pueden cambiar. Si cambian, alguna variable está usando el futuro (y en producción no lo tendría).


In [ ]:
corte_fuga = pd.Timestamp("2026-09-15")
recortadas = construir_features_tx(tx_evaluacion.loc[tx_evaluacion[FECHA] < corte_fuga], referencia)
diferencias = diferencias_entre_features(features, recortadas)
if diferencias:
    raise ValueError(f"data leakage check: expected 0 differences; observed {diferencias}")
print(f"Sin fuga de datos: {len(recortadas)} transacciones anteriores al corte tienen las mismas variables.")


## 4. Entrenar, comparar y elegir

Se entrena **solo con comportamiento normal** (mayo a julio). `LocalOutlierFactor` compara cada transacción nueva contra los datos de entrenamiento, así que se usa una **muestra reproducible** de hasta 20 000 filas.

Se comparan dos familias, con su corrida de MLflow cada una:

- **`iforest`** (*Isolation Forest*): el detector que casi todos prueban primero.
- **`lof`** (*Local Outlier Factor*), con distintos números de vecinos.

Y una **línea base sin aprendizaje**: marcar lo que supere el monto máximo que su tipo tuvo en el entrenamiento. Se elige con el **AP de validación** (promedio de precisión, que no depende del umbral).


In [ ]:
entrenamiento = features.loc[features[MES].isin(MESES_ENTRENAMIENTO)]
validacion = features.loc[features[MES] == MES_VALIDACION].reset_index(drop=True)
prueba = features.loc[features[MES] == MES_PRUEBA].reset_index(drop=True)
if int(entrenamiento[ETIQUETA_INYECTADA].sum()) != 0:
    raise ValueError("training months: expected no injected anomalies; observed some")
muestra = muestrear_entrenamiento(entrenamiento)
print({"entrenamiento": len(entrenamiento), "muestra": len(muestra), "validacion": len(validacion), "prueba": len(prueba)})

linea_base = regla_maximo_por_tipo(validacion, referencia)
metricas_base = evaluar_deteccion(validacion[ETIQUETA_INYECTADA], linea_base["score_anomalia"], linea_base["es_atipica"])
print("Línea base (máximo por tipo), validación:", {k: round(v, 3) for k, v in metricas_base.items() if k in ("auc", "average_precision", "precision", "recall")})


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

mlflow.set_experiment(nombres["experimento"])
CANDIDATOS = (("iforest", "iforest", 0), ("lof_20", "lof", 20), ("lof_50", "lof", 50), ("lof_100", "lof", 100), ("lof_200", "lof", 200))
filas_candidatos = []
for nombre, tipo, vecinos in CANDIDATOS:
    with mlflow.start_run(run_name=f"candidato_{nombre}"):
        detector = crear_detector(tipo, vecinos or 100).fit(muestra[list(FEATURES_TX)])
        puntaje = -detector.score_samples(validacion[list(FEATURES_TX)])
        auc = float(roc_auc_score(validacion[ETIQUETA_INYECTADA], puntaje))
        ap = float(average_precision_score(validacion[ETIQUETA_INYECTADA], puntaje))
        mlflow.log_params({"tipo_detector": tipo, "vecinos": vecinos, "filas_entrenamiento": len(muestra)})
        mlflow.log_metrics({"auc_validacion": auc, "average_precision_validacion": ap})
    filas_candidatos.append(
        {"nombre": nombre, "tipo": tipo, "vecinos": vecinos, "complejidad": vecinos,
         "auc_validacion": auc, "average_precision_validacion": ap}
    )
candidatos = pd.DataFrame(filas_candidatos)
elegido = elegir_candidato(candidatos)
display(candidatos.round(3))
print("Elegido (el menos complejo a menos de 0,01 de AP del mejor):", elegido)


**Qué leer en la tabla.** `iforest` rinde mucho peor aquí. No es un error: un *Isolation Forest* separa puntos con cortes al azar **dentro del rango que vio al entrenar**, y una madrugada o un monto muy alto caen **fuera de ese rango**, donde apenas se distinguen del borde. `lof` mide distancia a los vecinos y sí detecta lo que queda lejos. **La elección del algoritmo depende de cómo se ven las anomalías**, y eso se averigua con experimentos, no por reputación.


## 5. Fijar el umbral y medir en prueba

El modelo da un **puntaje**; qué tan alto debe ser para marcar depende de **cuántas se pueden revisar**. Se fija el umbral en el mes de **validación** para que se marque aproximadamente `p_tasa_revision`, y queda guardado **dentro del modelo** (`offset_`): el lote, `PREDICT` y el *endpoint* usan exactamente el mismo.

El mes de **prueba** se mide **una sola vez**, con el modelo final.


In [ ]:
elegida = candidatos.loc[candidatos["nombre"] == elegido].iloc[0]
modelo = crear_detector(elegida["tipo"], int(elegida["vecinos"]) or 100).fit(muestra[list(FEATURES_TX)])
umbral = fijar_umbral(modelo, validacion, tasa_revision)

puntos_validacion = puntuar_anomalias(modelo, validacion)
puntos_prueba = puntuar_anomalias(modelo, prueba)
metricas_validacion = evaluar_deteccion(validacion[ETIQUETA_INYECTADA], puntos_validacion["score_anomalia"], puntos_validacion["es_atipica"])
metricas_prueba = evaluar_deteccion(prueba[ETIQUETA_INYECTADA], puntos_prueba["score_anomalia"], puntos_prueba["es_atipica"])
base_prueba = regla_maximo_por_tipo(prueba, referencia)
metricas_base_prueba = evaluar_deteccion(prueba[ETIQUETA_INYECTADA], base_prueba["score_anomalia"], base_prueba["es_atipica"])

comparacion = pd.DataFrame(
    {"validación (umbral fijado)": metricas_validacion, "prueba (modelo)": metricas_prueba, "prueba (línea base)": metricas_base_prueba}
).T[["auc", "average_precision", "precision", "recall", "tasa_marcadas", "prevalencia"]]
display(comparacion.round(3))

por_tipo = recall_por_tipo(prueba, puntos_prueba["es_atipica"].to_numpy()).merge(
    recall_por_tipo(prueba, base_prueba["es_atipica"].to_numpy())[["tipo_anomalia", "recall"]],
    on="tipo_anomalia", suffixes=("_modelo", "_linea_base"),
)
display(por_tipo.round(2))


**Cómo leerlo.**

- La **línea base** tiene buena precisión (marca poco y casi siempre acierta), pero **solo ve montos altos**: las ráfagas le son invisibles y deja pasar la mayoría de las anomalías.
- El **modelo** combina varias variables y ve los tres tipos, a cambio de más marcas falsas. **Ninguno gana en todos los tipos**: la línea base detecta mejor los montos extremos. En la práctica se **combinan** reglas y modelo.
- La **tasa de marcadas** en prueba difiere un poco de `p_tasa_revision`: el umbral se fijó en agosto. Esa diferencia es justo lo que el monitoreo (sección 9) vigila.
- Con `p_tasa_revision = 1 %`, una precisión cercana a 50 % significa que **de cada dos transacciones que el equipo revisa, una es realmente atípica**. Esa es la conversación de negocio.


## 6. Registrar el modelo con su firma y su referencia

La corrida final guarda el modelo con su **firma** (8 columnas de entrada y una de salida), las métricas y la **referencia** como artefacto: así, quien consuma el modelo sabe con qué referencia se calcularon sus variables. La versión se registra con etiquetas.

**La salida se declara como una columna** (`pd.Series`), no como un arreglo de NumPy. Un arreglo se infiere como **tensor**, y entonces la página de la versión dice «Endpoints aren't supported for this model… has tensors» y deshabilita **Activate version endpoint**.


In [ ]:
validar_parametros_ml(notebookutils.runtime.context, p_lakehouse_esperado, p_prefijo, p_confirmar_reemplazo)
referencia = calcular_referencia(transacciones_entrenamiento, fecha_utc=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
muestra_firma = muestra[list(FEATURES_TX)].head(1000)
with mlflow.start_run(run_name=f"final_{elegido}") as corrida:
    firma = infer_signature(muestra_firma, pd.Series(modelo.predict(muestra_firma), name="prediccion"))
    mlflow.sklearn.log_model(modelo, "model", signature=firma, serialization_format="cloudpickle")
    mlflow.log_dict({"referencia": referencia.to_dict("records"), "features": list(FEATURES_TX)}, "referencia_tx.json")
    mlflow.log_params({
        "tipo_detector": elegida["tipo"], "vecinos": int(elegida["vecinos"]), "tasa_revision": tasa_revision,
        "umbral_offset": round(umbral, 6), "filas_entrenamiento": len(muestra),
    })
    mlflow.log_metrics({
        **{f"{k}_validacion": v for k, v in metricas_validacion.items()},
        **{f"{k}_prueba": v for k, v in metricas_prueba.items()},
    })
    run_id = corrida.info.run_id
registrada = mlflow.register_model(
    f"runs:/{run_id}/model",
    nombres["modelo"],
    tags={
        "tipo_detector": elegida["tipo"],
        "tasa_revision": str(tasa_revision),
        "mes_prueba": MES_PRUEBA,
        "auc_prueba": f"{metricas_prueba['auc']:.4f}",
        "evaluacion": "anomalias_inyectadas",
    },
)
version = int(registrada.version)
print(f"Modelo {nombres['modelo']}, versión {version}; columnas de entrada en la firma: {len(firma.inputs.input_names())}")


### La versión cargada es el mismo modelo

Cargar `models:/<nombre>/<versión>` es lo que harán otros notebooks, pipelines y el *endpoint*. Debe dar **exactamente** los mismos puntajes y las mismas marcas, incluido el umbral.


In [ ]:
modelo_registrado = mlflow.sklearn.load_model(f"models:/{nombres['modelo']}/{version}")
muestra_prueba = prueba.head(2000)
if not np.allclose(modelo.decision_function(muestra_prueba[list(FEATURES_TX)]), modelo_registrado.decision_function(muestra_prueba[list(FEATURES_TX)])):
    raise ValueError("registry roundtrip: expected identical scores; observed differences")
if not np.array_equal(modelo.predict(muestra_prueba[list(FEATURES_TX)]), modelo_registrado.predict(muestra_prueba[list(FEATURES_TX)])):
    raise ValueError("registry roundtrip: expected identical flags; observed differences")
print("La versión registrada reproduce los puntajes y las marcas (el umbral viaja dentro del modelo).")


## 7. Puntuar por lotes

Se puntúa septiembre y se guardan dos tablas: la **referencia** (nueve filas) y las **transacciones puntuadas**, con la versión del modelo y la fecha para poder auditar qué modelo marcó qué. La escritura reemplaza solo las filas de este mes, así que se puede repetir sin duplicar.


In [ ]:
puntuadas = puntuar_anomalias(modelo_registrado, prueba)
detalle = prueba[[CLAVE_TX, CUENTA, MES, FECHA, "tipo_transaccion", "canal", "monto", ETIQUETA_INYECTADA, TIPO_INYECTADO]].merge(puntuadas, on=CLAVE_TX)
detalle[FECHA] = detalle[FECHA].dt.strftime("%Y-%m-%d %H:%M:%S")
detalle["version_modelo"] = version
detalle["fecha_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
filas_referencia = escribir_tabla(spark, nombres["referencia"], referencia, COLUMNAS_REFERENCIA_TX)
filas_puntuadas = reemplazar_corte(spark, nombres["anomalias"], detalle, COLUMNAS_ANOMALIAS_TX, MES, MES_PRUEBA)
print(json.dumps({"status": "success", "referencia": filas_referencia, "puntuadas": filas_puntuadas, "version": version}))
display(detalle.sort_values("score_anomalia", ascending=False).head(10))


### PREDICT: el mismo resultado con Spark

`PREDICT` aplica el modelo registrado a un DataFrame de Spark. Debe dar **la misma marca** que la puntuación con pandas (`-1` = atípica, `1` = normal). Los sabores soportados incluyen `sklearn`, y la firma es obligatoria.


In [ ]:
try:
    from synapse.ml.predict import MLFlowTransformer
except ImportError as error:
    print(f"MLFlowTransformer no disponible en este entorno: {error}")
else:
    entrada = spark.createDataFrame(prueba[[CLAVE_TX, *FEATURES_TX]])
    transformador = MLFlowTransformer(
        inputCols=list(FEATURES_TX), outputCol="prediccion_predict",
        modelName=nombres["modelo"], modelVersion=version,
    )
    predict = transformador.transform(entrada).select(CLAVE_TX, "prediccion_predict").toPandas()
    union = puntuadas.merge(predict, on=CLAVE_TX)
    acuerdo = float(((union["es_atipica"] == 1) == (union["prediccion_predict"].astype(int) == -1)).mean())
    print(f"Acuerdo entre pandas y PREDICT: {acuerdo:.4f} ({len(union)} transacciones)")
    if acuerdo < 1.0:
        raise ValueError(f"PREDICT: expected agreement 1.0; observed {acuerdo}")


## 8. Puntuar en línea: una transacción por el *endpoint*

En línea, una aplicación recibe **una** transacción y necesita decidir al instante. Calcula sus variables y llama al *endpoint*. Lo importante es **cómo se calculan esas variables**: con `features_de_una_transaccion`, que usa el **mismo código** que el entrenamiento, el historial previo de la cuenta y la referencia.

Aquí se arman dos solicitudes para una cuenta real: una transacción **normal** y una **sospechosa** (una compra por cajero automático a las 3 a. m. con un monto seis veces el habitual).


In [ ]:
cuenta_ejemplo = tx_evaluacion.loc[tx_evaluacion[ETIQUETA_INYECTADA] == 0].groupby(CUENTA).size().loc[lambda s: s >= 20].index[0]
historial = tx_evaluacion.loc[(tx_evaluacion[CUENTA] == cuenta_ejemplo) & (tx_evaluacion[ETIQUETA_INYECTADA] == 0) & (tx_evaluacion[FECHA] < pd.Timestamp("2026-09-20"))]
ultima = historial.sort_values(FECHA).iloc[-1]
habitual = float(historial.loc[historial["tipo_transaccion"] == "COMPRA", "monto"].median())
nuevas = {
    "normal": {CLAVE_TX: "TXNUEVA01", CUENTA: cuenta_ejemplo, FECHA: pd.Timestamp("2026-09-21 11:30:00"),
               "tipo_transaccion": "COMPRA", "direccion": "DEBITO", "canal": ultima["canal"], "monto": habitual},
    "sospechosa": {CLAVE_TX: "TXNUEVA02", CUENTA: cuenta_ejemplo, FECHA: pd.Timestamp("2026-09-21 03:05:00"),
                   "tipo_transaccion": "COMPRA", "direccion": "DEBITO", "canal": "ATM", "monto": round(habitual * 6, 2)},
}
solicitudes = {nombre: features_de_una_transaccion(historial, nueva, referencia) for nombre, nueva in nuevas.items()}
variables_en_linea = pd.concat(solicitudes.values(), ignore_index=True)
display(variables_en_linea[[CLAVE_TX, *FEATURES_TX]].round(3))
cuerpo = solicitud_endpoint(variables_en_linea)
print("Cuerpo de la solicitud (formato del endpoint):")
print(json.dumps(cuerpo, ensure_ascii=False)[:600])
local = puntuar_anomalias(modelo_registrado, variables_en_linea)
display(local.assign(descripcion=list(nuevas)).round(3))


**Llamar al *endpoint* (vista previa).** Un modelo nuevo **nunca** tiene su *endpoint* activo: se activa a propósito, desde la página de la versión (**Activate version endpoint**) o por la API, y tarda unos minutos. Esta celda **no activa nada**: busca la versión activa más reciente del modelo, le envía la solicitud, mide el tiempo y comprueba que la respuesta coincida con la de **esa misma versión** cargada aquí. Si no hay ninguna activa, lo dice y sigue.

**Cómo probarla:** activa una versión, espera a que quede `Active` y **vuelve a ejecutar solo esta celda**. Recuerda lo documentado: sabores `sklearn`, LightGBM, XGBoost y Keras con firma; se **duerme a los 5 minutos** sin uso (y la primera llamada tarda más); consume capacidad mientras está activo.


In [ ]:
if p_probar_endpoint != "SI":
    print("Prueba del endpoint omitida (p_probar_endpoint distinto de 'SI').")
else:
    try:
        import sempy.fabric as fabric
        modelos = fabric.list_items(type="MLModel")
        id_modelo = modelos.loc[modelos["Display Name"] == nombres["modelo"], "Id"].iloc[0]
        respuesta_endpoint = puntuar_en_linea(
            fabric.FabricRestClient(), notebookutils.runtime.context["currentWorkspaceId"], id_modelo, cuerpo
        )
    except Exception as error:  # el endpoint es una vista previa: puede no estar disponible en tu tenant
        respuesta_endpoint = {"estado": f"no disponible ({type(error).__name__}: {str(error)[:120]})", "version": None, "predicciones": None}
    print("Endpoint del modelo", nombres["modelo"], "→", respuesta_endpoint["estado"])
    if respuesta_endpoint["predicciones"] is None:
        print("Para probarlo: activa una versión en su página (Activate version endpoint) y vuelve a ejecutar esta celda.")
    else:
        activo = mlflow.sklearn.load_model(f"models:/{nombres['modelo']}/{respuesta_endpoint['version']}")
        esperado = [int(m) for m in activo.predict(variables_en_linea[list(FEATURES_TX)])]
        print("Versión", respuesta_endpoint["version"], "| marcas del endpoint (-1 = atípica):", respuesta_endpoint["predicciones"], "| tiempo de la llamada:", round(respuesta_endpoint["segundos"], 2), "s")
        if respuesta_endpoint["predicciones"] != esperado:
            raise ValueError(f"endpoint: expected {esperado}; observed {respuesta_endpoint['predicciones']}")
        print("El endpoint coincide con la puntuación local de la misma versión.")


## 9. Monitorear sin etiquetas

Los casos confirmados llegan tarde (o nunca). Mientras tanto hay dos señales que **no necesitan etiquetas**:

1. **La tasa de marcadas.** Si el umbral se fijó para marcar el 1 %, una tasa que se duplica o desaparece indica un cambio.
2. **La deriva de las variables** (PSI, igual que en B4) entre el entrenamiento y el mes nuevo.

Para ver que funcionan se **simula** un cambio de comportamiento: en septiembre, todos los montos suben 60 % (por ejemplo, por inflación o una campaña). Las variables se recalculan y se vuelven a puntuar.


In [ ]:
def reporte_deriva_tx(referencia_df: "pd.DataFrame", actual_df: "pd.DataFrame") -> "pd.DataFrame":
    filas = [{"variable": c, "psi": psi_numerico(referencia_df[c], actual_df[c])} for c in FEATURES_NUMERICAS_TX]
    filas += [{"variable": c, "psi": psi_categorico(referencia_df[c], actual_df[c])} for c in FEATURES_CATEGORICAS_TX]
    resultado = pd.DataFrame(filas)
    resultado["deriva"] = resultado["psi"].map(clasificar_psi)
    return resultado.sort_values("psi", ascending=False).reset_index(drop=True)


tx_simulada = tx_evaluacion.copy()
en_prueba = tx_simulada[FECHA].dt.strftime("%Y-%m") == MES_PRUEBA
tx_simulada.loc[en_prueba, "monto"] = (tx_simulada.loc[en_prueba, "monto"] * 1.6).round(2)
prueba_simulada = construir_features_tx(tx_simulada, referencia)
prueba_simulada = prueba_simulada.loc[prueba_simulada[MES] == MES_PRUEBA].reset_index(drop=True)

marcadas_por_mes = pd.DataFrame(
    [
        {"escenario": "validación (donde se fijó el umbral)", "tasa_marcadas": float(puntos_validacion["es_atipica"].mean())},
        {"escenario": "prueba (septiembre)", "tasa_marcadas": float(puntos_prueba["es_atipica"].mean())},
        {"escenario": "prueba con montos +60 % (simulado)", "tasa_marcadas": float(puntuar_anomalias(modelo_registrado, prueba_simulada)["es_atipica"].mean())},
    ]
)
display(marcadas_por_mes.round(4))
deriva_normal = reporte_deriva_tx(muestra, prueba)
deriva_simulada = reporte_deriva_tx(muestra, prueba_simulada)
display(deriva_normal.merge(deriva_simulada, on="variable", suffixes=("_septiembre", "_simulado")).round(3))


**Qué leer.** Con datos normales las variables quedan `estable` y la tasa de marcadas se mantiene cerca de la capacidad de revisión. Con el cambio simulado, **`z_monto_tipo` deriva** y la tasa de marcadas **se dispara**: el sistema avisa aunque no haya una sola etiqueta. Qué hacer con el aviso (revisar la referencia, reentrenar con datos nuevos) es una decisión de una persona, igual que en B4.

Fabric no trae un monitor de deriva integrado: esta medición vive en tablas y notebooks propios, y las alertas se pueden enviar con **Activator** o desde un pipeline.


## 10. Llevarlo a producción: qué falta

| Pieza | En este bonus | En un proceso real |
|---|---|---|
| Etiquetas | Anomalías inyectadas | Casos que el equipo de revisión confirma o descarta; con ellas se mide precisión real |
| Referencia y variables | Función compartida + tabla de referencia | Un almacén de variables o un servicio que calcule las mismas variables en línea |
| Umbral | Capacidad de revisión fija (1 %) | Se revisa con la precisión observada y el costo de una revisión |
| Ciclo mensual | Se ejecuta a mano | Se programa como en B4 (campeón, retador, regla de promoción, reversión) |
| Endpoint | Se activa a mano | Se activa en el despliegue; se mide su latencia con la carga real antes de comprometer un SLA |

**Siguiente paso.** Reemplazar el dataset sintético por tus transacciones, mantener la misma estructura (referencia, variables sin fuga, evaluación temporal, umbral por capacidad) y medir con los casos que ya conozcas.
